In [ ]:
# Download the repository
!git clone -q https://github.com/yasahi-hpc/JAX-PyTorch-Vlasov.git

# Make the source directory importable
import sys
sys.path.insert(0, "JAX-PyTorch-Vlasov/simulations/heat3d/jax/src")

In [ ]:
!pip install "xarray[complete]"

In [ ]:
import jax.numpy as jnp
import os
from heat3d import (
    run_heat3d
)

nbiter, diag_steps = 20, 40
device = 'tpu'
out_dir = 'data_python'
lx = 2.0 * jnp.pi
dt = 0.001
kappa = 1.0
nrepeats = 1

# Mapping of local directories to their desired names inside the zip
device_name = 'TPUv5' if device == 'tpu' else 'cpu'

for dtype in ["float32"]:
    for nx in [32, 64, 128, 256, 512]:
        for solver_type in [0, 1, 2]:
            # Run simulation
            run_heat3d(
                nx=nx,
                lx=lx,
                nbiter=nbiter,
                nrepeats=nrepeats,
                diag_steps=diag_steps,
                dt=dt,
                out_dir=out_dir,
                kappa=kappa,
                solver_type=solver_type,
                dtype=dtype,
            )

            # Rename the resulting file to the requested format
            old_filename = f'heat3d_{dtype}.txt'
            new_filename = f'heat3d_{device_name}_{dtype}_N{nx}_solver{solver_type}.txt'
            if os.path.exists(old_filename):
                os.rename(old_filename, new_filename)
                print(f'Renamed {old_filename} to {new_filename}')

In [ ]:
import zipfile
import os
from google.colab import files

# Define the name of the output zip file
zip_filename = 'heat3d_results.zip'

# Mapping of local directories to their desired names inside the zip
dump_mapping = {
    'jaxpr_dump': f'heat3d_{device_name}_{dtype}_jaxpr_dump',
    'hlo_dump': f'heat3d_{device_name}_{dtype}_hlo_dump'
}

# Create a zip archive
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    # Add .txt files from the current directory
    txt_files = [f for f in os.listdir('.') if f.startswith('heat3d_') and f.endswith('.txt')]
    for file in txt_files:
        zipf.write(file)

    # Add files from dump folders into renamed subdirectories
    for local_folder, zip_folder in dump_mapping.items():
        if os.path.exists(local_folder):
            for root, dirs, files_in_dir in os.walk(local_folder):
                for file in files_in_dir:
                    file_path = os.path.join(root, file)
                    # Construct the internal path: new_folder_name/filename
                    archive_path = os.path.join(zip_folder, file)
                    zipf.write(file_path, arcname=archive_path)

# Download the zip file
files.download(zip_filename)